In [1]:
import os
import sys
import scipy
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import scipy.io as sio
import scanpy.external as sce
import matplotlib.pyplot as plt
import re
import anndata as ad
import statistics
import torch
import scvi
import tempfile
import sklearn
from scib_metrics.benchmark import Benchmarker
from scvi.model.utils import mde
from tqdm import tqdm
print("Last run with scvi-tools version:", scvi.__version__)
scvi.settings.num_threads = 4
scvi.settings.seed = 0
sc._settings.ScanpyConfig.n_jobs=72
sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, fontsize=10, dpi_save=400,
    facecolor = 'white', figsize=(8,8), format='png')
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)

/home/liyanguo/anaconda3/envs/R/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/liyanguo/anaconda3/envs/R/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/home/liyanguo/anaconda3/envs/R/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)
/home/liyanguo/anaconda3/envs/R/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing CSCDataset from `anndata.experimental` is deprecated. Import anndata.abc.CSCDataset instead.
  warnings.warn(msg, FutureWarning)
/home/liyanguo/anaconda3/envs/R/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Impo

Last run with scvi-tools version: 1.2.0


In [2]:
sc.logging.print_header()

Package,Version
scipy,1.15.2
numpy,1.26.4
pandas,2.2.3
scanpy,1.11.0
seaborn,0.13.2
matplotlib,3.9.2
anndata,0.11.3
torch,2.4.1 (2.4.1+cu121)
scvi-tools,1.2.0
scikit-learn,1.5.2


# 1. Got HVG and scVI data for L1 refine

In [3]:
dataset = "Ref_Atlas"

In [4]:
obj_path = f'/home/liyanguo/MyImmuCell/05_Ref_Atlas_subpopulation/Level2_Refine_R1/'

Celltype_L1_L2
CEACAM8- Neutrophil       242504
CD4+ T                    169312
NK                        126749
CD8+ T                    121700
Classical monocyte         86519
B|Plasma                   50422
MAIT                       22314
Non-classical monocyte     21635
γδ T                       14576
Platelet                   12348
Dendritic                   4730
CEACAM8+ Neutrophil         3398
Basophil                    3010
Proliferative T/NK          2529
pDC                         2492
HSPC                         389
iNKT                          44

In [5]:
celltypes=['CEACAM8_Neg_Neutrophil','CD4T','NK',
           'CD8T','Monocyte','B','gdT','Proliferative_TNK',
           'MAIT','Platelet','Basophil',
           'Dendritic','CEACAM8_Pos_Neutrophil',
           'pDC']

In [6]:
for celltype in celltypes:
    adata = sc.read_h5ad(f"{obj_path}{celltype}/{celltype}.h5ad")
    adata.layers["counts"] = adata.X.copy()
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata,flavor='seurat_v3',n_top_genes=2000,layer="counts",
                            subset=True)
    
    ##　Save scRNA for scVI
    adata.write(f"{obj_path}{celltype}/{celltype}_preprocess_scRNA.h5ad",compression="gzip")
    print(f"{celltype} Done!")

CEACAM8_Neg_Neutrophil Done!
CD4T Done!
NK Done!
CD8T Done!
Monocyte Done!
B Done!
gdT Done!
Proliferative_TNK Done!
MAIT Done!
Platelet Done!
Basophil Done!
Dendritic Done!
CEACAM8_Pos_Neutrophil Done!
pDC Done!


In [7]:
# Too few cells < 1000

In [8]:
celltypes=['HSPC','iNKT']

In [9]:
for celltype in celltypes:
    adata = sc.read_h5ad(f"{obj_path}{celltype}/{celltype}.h5ad")
    adata.layers["counts"] = adata.X.copy()
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    
    ##　Save scRNA for scVI
    adata.write(f"{obj_path}{celltype}/{celltype}_preprocess_scRNA.h5ad",compression="gzip")
    print(f"{celltype} Done!")

HSPC Done!
iNKT Done!
